# Soil Quality Classification — Dataset Exploration

This notebook follows the same dataset-exploration style as the placement prediction practical, adapted for the Soil Quality Excel dataset.

It covers:
- Dataset loading
- Shape and columns
- Basic information
- Missing values
- Duplicate rows
- Duplicate removal
- Unique values
- Target distribution
- Descriptive statistics
- Numerical and categorical columns
- Rows containing missing values
- Basic visual exploration
- Missing-value treatment
- Outlier detection
- Categorical encoding
- Train/test split
- SVM and KNN preparation
- Model training and evaluation
- Confusion matrices and model comparison


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


In [ ]:
# Load the Soil Quality Excel dataset
file_path = 'Soil_Quality_SVM_KNN_12500_Incomplete(1).xlsx'
df = pd.read_excel(file_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)


In [ ]:
# Display first 5 rows
df.head()


In [ ]:
# Display last 5 rows
df.tail()


In [ ]:
# Dataset columns
print("Column names:")
print(df.columns.tolist())


In [ ]:
# Dataset information
df.info()


In [ ]:
# Check dataset shape
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])


In [ ]:
# Check missing values
missing_values = df.isnull().sum()
print("Missing values in each column:")
print(missing_values)

print("\nTotal missing cells:", df.isnull().sum().sum())


In [ ]:
# Missing-value percentage
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_table = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Missing Percentage': missing_percentage
})
missing_table


In [ ]:
# Check duplicate rows
duplicate_rows = df[df.duplicated()]

print(f"Total duplicate rows detected: {len(duplicate_rows)}")
print(duplicate_rows.head(20))


In [ ]:
# Number of duplicate rows
df.duplicated().sum()


In [ ]:
# Remove duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", df.shape)


In [ ]:
# Unique values in each column
for column in df.columns:
    print(f"\n{column}:")
    print(df[column].unique()[:20])
    if df[column].nunique(dropna=True) > 20:
        print("... (showing first 20 unique values)")


In [ ]:
# Soil Quality class distribution
print(df['Soil_Quality'].value_counts(dropna=False))


In [ ]:
# Soil Quality class percentages
print((df['Soil_Quality'].value_counts(normalize=True, dropna=False) * 100).round(2))


In [ ]:
# Descriptive statistics for numerical columns
df.describe()


In [ ]:
# Numerical columns
numerical_columns = df.select_dtypes(include=np.number).columns.tolist()
print("Numerical columns:")
print(numerical_columns)


In [ ]:
# Categorical columns
categorical_columns = df.select_dtypes(include='object').columns.tolist()
print("Categorical columns:")
print(categorical_columns)


In [ ]:
# Rows containing at least one missing value
rows_with_missing = df[df.isnull().any(axis=1)]

print("Number of rows with missing values:", len(rows_with_missing))
rows_with_missing.head(20)


## Visual Exploration

In [ ]:
# Target class distribution
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x='Soil_Quality')
plt.title('Soil Quality Class Distribution')
plt.xlabel('Soil Quality')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
# Histograms for numerical features
df[numerical_columns].hist(figsize=(14, 12), bins=30)
plt.suptitle('Distribution of Numerical Soil Features')
plt.tight_layout()
plt.show()


In [ ]:
# Boxplots for numerical features
for column in numerical_columns:
    plt.figure(figsize=(7, 4))
    sns.boxplot(x=df[column])
    plt.title(f'Boxplot of {column}')
    plt.tight_layout()
    plt.show()


In [ ]:
# Correlation heatmap
plt.figure(figsize=(11, 8))
sns.heatmap(df[numerical_columns].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap of Numerical Features')
plt.tight_layout()
plt.show()


## Missing-Value Treatment

In [ ]:
# Make a working copy for preprocessing
data = df.copy()

# Separate target from input features
X = data.drop(columns=['Soil_Quality'])
y = data['Soil_Quality']

print("Feature shape:", X.shape)
print("Target shape:", y.shape)


In [ ]:
# Identify feature types
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include='object').columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


In [ ]:
# Preprocessing:
# - Numeric missing values -> median
# - Categorical missing values -> most frequent
# - Categorical columns -> one-hot encoding
# - Numeric columns -> standard scaling

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)


## Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


## SVM Model

In [ ]:
svm_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', SVC(kernel='rbf', random_state=42))
])

svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)

svm_accuracy = accuracy_score(y_test, y_pred_svm)

print("SVM Accuracy:", round(svm_accuracy * 100, 2), "%")
print("\nSVM Classification Report:")
print(classification_report(y_test, y_pred_svm))


In [ ]:
# SVM confusion matrix
cm_svm = confusion_matrix(y_test, y_pred_svm, labels=svm_model.classes_)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_svm,
    annot=True,
    fmt='d',
    xticklabels=svm_model.classes_,
    yticklabels=svm_model.classes_
)
plt.title('SVM Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()


## KNN Model

In [ ]:
knn_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier(n_neighbors=5))
])

knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_test)

knn_accuracy = accuracy_score(y_test, y_pred_knn)

print("KNN Accuracy:", round(knn_accuracy * 100, 2), "%")
print("\nKNN Classification Report:")
print(classification_report(y_test, y_pred_knn))


In [ ]:
# KNN confusion matrix
cm_knn = confusion_matrix(y_test, y_pred_knn, labels=knn_model.classes_)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_knn,
    annot=True,
    fmt='d',
    xticklabels=knn_model.classes_,
    yticklabels=knn_model.classes_
)
plt.title('KNN Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()


## Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': ['SVM', 'KNN'],
    'Accuracy': [svm_accuracy, knn_accuracy]
})

comparison['Accuracy (%)'] = (comparison['Accuracy'] * 100).round(2)
comparison


In [ ]:
# Accuracy comparison graph
plt.figure(figsize=(7, 5))
sns.barplot(data=comparison, x='Model', y='Accuracy (%)')
plt.title('SVM vs KNN Accuracy')
plt.ylabel('Accuracy (%)')
plt.ylim(0, 100)
plt.tight_layout()
plt.show()


## Notes

- The notebook uses the Soil Quality Excel file as the dataset source.
- Duplicate rows are identified and removed before modeling.
- Missing values are handled inside the preprocessing pipeline.
- Numerical features are standardized because distance-based KNN and SVM can be affected by feature scale.
- Categorical features such as `Soil_Type` and `Crop_Type` are one-hot encoded.
- The final section compares SVM and KNN using test-set accuracy and classification reports.
